Taxa de aprovação por escola

In [0]:
-- Qual escola tem maior taxa de aprovação?
SELECT
    escola,
    COUNT(*) AS total_alunos,
    SUM(CASE WHEN status_aprovacao = 'aprovado' THEN 1 ELSE 0 END) AS total_aprovados,
    SUM(CASE WHEN status_aprovacao = 'reprovado' THEN 1 ELSE 0 END) AS total_reprovados,
    ROUND(
        SUM(CASE WHEN status_aprovacao = 'aprovado' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        1
    ) AS taxa_aprovacao_pct
FROM workspace.lakehouse_edu.silver_student_performance
GROUP BY escola
ORDER BY taxa_aprovacao_pct DESC

Impacto das horas de estudos na nota final


In [0]:
-- Horas de estudo impactam a nota final?
SELECT
    horas_estudo_semana,
    COUNT(*) AS total_alunos,
    ROUND(AVG(nota_final), 2) AS media_nota_final,
    ROUND(AVG(faltas), 1) AS media_faltas,
    SUM(CASE WHEN status_aprovacao = 'aprovado' THEN 1 ELSE 0 END) AS total_aprovados,
    ROUND(
        SUM(CASE WHEN status_aprovacao = 'aprovado' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        1
    ) AS taxa_aprovacao_pct
FROM workspace.lakehouse_edu.silver_student_performance
WHERE status_aprovacao != 'sem_avaliacao'
GROUP BY horas_estudo_semana
ORDER BY horas_estudo_semana

Top 10 alunos por desempenho

In [0]:
-- Top 10 alunos por desempenho
SELECT
    escola,
    sexo,
    idade,
    tipo_endereco,
    horas_estudo_semana,
    reprovacoes_anteriores,
    nota_1_bimestre,
    nota_2_bimestre,
    nota_final,
    media_notas,
    faixa_desempenho,
    ranking
FROM workspace.lakehouse_edu.gold_perfil_alunos
WHERE ranking <= 10
ORDER BY ranking

Alunos com risco de reprovação

In [0]:
-- Perfil de alunos com maior risco de reprovação
SELECT
    escola,
    sexo,
    idade,
    tipo_endereco,
    reprovacoes_anteriores,
    horas_estudo_semana,
    tem_internet,
    quer_ensino_superior,
    qualidade_relacao_familiar,
    tempo_livre,
    frequencia_saidas,
    nota_1_bimestre,
    nota_2_bimestre,
    nota_final,
    faltas,
    status_aprovacao
FROM workspace.lakehouse_edu.silver_student_performance
WHERE reprovacoes_anteriores > 0
  AND nota_final < 10
  AND status_aprovacao = 'reprovado'
ORDER BY reprovacoes_anteriores DESC, nota_final ASC

Análise de progressão das notas

In [0]:
-- Como as notas evoluem ao longo das avaliações?
SELECT
    status_aprovacao,
    COUNT(*) AS total_alunos,
    ROUND(AVG(nota_1_bimestre), 2) AS media_1_bimestre,
    ROUND(AVG(nota_2_bimestre), 2) AS media_2_bimestre,
    ROUND(AVG(nota_final), 2) AS media_nota_final,
    ROUND(AVG(nota_final) - AVG(nota_1_bimestre), 2) AS variacao_1_para_final,
    ROUND(AVG(faltas), 1) AS media_faltas
FROM workspace.lakehouse_edu.silver_student_performance
WHERE status_aprovacao != 'sem_avaliacao'
GROUP BY status_aprovacao
ORDER BY media_nota_final DESC

Perfil socioeconômico e desempenho

In [0]:
-- Perfil socioeconômico impacta o desempenho?
SELECT
    escolaridade_mae,
    COUNT(*) AS total_alunos,
    ROUND(AVG(nota_final), 2) AS media_nota_final,
    ROUND(AVG(horas_estudo_semana), 2) AS media_horas_estudo,
    ROUND(AVG(faltas), 1) AS media_faltas,
    SUM(CASE WHEN status_aprovacao = 'aprovado' THEN 1 ELSE 0 END) AS total_aprovados,
    ROUND(
        SUM(CASE WHEN status_aprovacao = 'aprovado' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        1
    ) AS taxa_aprovacao_pct,
    SUM(CASE WHEN quer_ensino_superior = 'yes' THEN 1 ELSE 0 END) AS quer_ensino_superior
FROM workspace.lakehouse_edu.silver_student_performance
WHERE status_aprovacao != 'sem_avaliacao'
GROUP BY escolaridade_mae
ORDER BY escolaridade_mae

Resumo executivo do Lakehouse

In [0]:
-- Resumo executivo — visão geral do dataset
SELECT
    COUNT(*) AS total_alunos,
    COUNT(DISTINCT escola) AS total_escolas,
    ROUND(AVG(nota_final), 2) AS media_geral_notas,
    ROUND(AVG(faltas), 1) AS media_geral_faltas,
    SUM(CASE WHEN status_aprovacao = 'aprovado' THEN 1 ELSE 0 END) AS total_aprovados,
    SUM(CASE WHEN status_aprovacao = 'reprovado' THEN 1 ELSE 0 END) AS total_reprovados,
    SUM(CASE WHEN status_aprovacao = 'sem_avaliacao' THEN 1 ELSE 0 END) AS total_sem_avaliacao,
    ROUND(
        SUM(CASE WHEN status_aprovacao = 'aprovado' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        1
    ) AS taxa_aprovacao_geral_pct,
    ROUND(
        SUM(CASE WHEN status_aprovacao = 'reprovado' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        1
    ) AS taxa_reprovacao_geral_pct,
    ROUND(
        SUM(CASE WHEN quer_ensino_superior = 'yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        1
    ) AS pct_quer_ensino_superior
FROM workspace.lakehouse_edu.silver_student_performance